# 15 — Advanced Transformer Engineering: Efficient Attention, RoPE, RMSNorm, KV Cache Skeleton

Goal: build modern decoder-only blocks closer to production GPT-style models.

_Generated: 2026-01-25_

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
pip install safetensors sentencepiece
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. Modern decoder block components (practical)

Common choices in modern decoder-only LMs:
- Pre-norm residual blocks (norm before attention/MLP)
- RMSNorm instead of LayerNorm (often)
- GELU/SwiGLU MLP
- Rotary Positional Embeddings (RoPE)
- Causal self-attention with KV-cache for generation
- Dropout in training; none in inference

This notebook provides **educational implementations** you can adapt.

In [ ]:

import torch, torch.nn as nn
import torch.nn.functional as F
import math

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(d))
    def forward(self, x):
        # x: [..., d]
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.scale

def rotate_half(x):
    x1, x2 = x[..., :x.size(-1)//2], x[..., x.size(-1)//2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(q, k, sin, cos):
    # q,k: [B, H, T, Dh]
    q = (q * cos) + (rotate_half(q) * sin)
    k = (k * cos) + (rotate_half(k) * sin)
    return q, k

class RoPE(nn.Module):
    def __init__(self, dim, max_len=2048, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_len).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)  # [T, dim/2]
        emb = torch.cat([freqs, freqs], dim=-1)       # [T, dim]
        self.register_buffer("sin", emb.sin()[None, None, :, :])  # [1,1,T,dim]
        self.register_buffer("cos", emb.cos()[None, None, :, :])  # [1,1,T,dim]
    def forward(self, T):
        return self.sin[:, :, :T, :], self.cos[:, :, :T, :]

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, rope: RoPE=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3*d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)
        self.rope = rope

    def forward(self, x, kv_cache=None):
        # x: [B, T, d]
        B, T, d = x.shape
        qkv = self.qkv(x)  # [B, T, 3d]
        q, k, v = qkv.chunk(3, dim=-1)
        # to heads: [B, H, T, Dh]
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1,2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1,2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1,2)

        if self.rope is not None:
            sin, cos = self.rope(T)
            q, k = apply_rope(q, k, sin, cos)

        # KV-cache: append keys/values for generation
        if kv_cache is not None:
            k_prev, v_prev = kv_cache
            k = torch.cat([k_prev, k], dim=2)
            v = torch.cat([v_prev, v], dim=2)

        # causal attention: compute scores for each head
        # q: [B,H,Tq,Dh], k: [B,H,Tk,Dh]
        Tk = k.size(2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)  # [B,H,T,Tk]
        causal = torch.triu(torch.ones(T, Tk, device=x.device), diagonal=1 + (Tk - T)).bool()
        att = att.masked_fill(causal, float("-inf"))
        probs = torch.softmax(att, dim=-1)
        y = probs @ v  # [B,H,T,Dh]
        y = y.transpose(1,2).contiguous().view(B, T, d)
        y = self.out(y)

        new_cache = (k, v) if kv_cache is not None else None
        return y, new_cache

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_model, d_ff, bias=False)
        self.w3 = nn.Linear(d_ff, d_model, bias=False)
    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, rope=None):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, rope=rope)
        self.norm2 = RMSNorm(d_model)
        self.mlp = SwiGLU(d_model, d_ff)
    def forward(self, x, kv_cache=None):
        a, new_cache = self.attn(self.norm1(x), kv_cache=kv_cache)
        x = x + a
        m = self.mlp(self.norm2(x))
        x = x + m
        return x, new_cache

## 2. Assemble a decoder-only LM using the advanced block

In [ ]:

class GPTMini(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=8, n_layers=4, d_ff=768, max_len=512, pad_id=0):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.rope = RoPE(dim=d_model//n_heads, max_len=max_len)
        self.blocks = nn.ModuleList([DecoderBlock(d_model, n_heads, d_ff, rope=self.rope) for _ in range(n_layers)])
        self.norm = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids, kv_cache=None):
        # kv_cache: list[(k,v)] per layer for generation; None for training
        x = self.tok(input_ids)
        new_cache = [] if kv_cache is not None else None
        for i, blk in enumerate(self.blocks):
            layer_cache = kv_cache[i] if kv_cache is not None else None
            x, c = blk(x, kv_cache=layer_cache)
            if kv_cache is not None:
                new_cache.append(c)
        x = self.norm(x)
        logits = self.head(x)
        return logits, new_cache

# quick shape check
vocab_size = 128
m = GPTMini(vocab_size=vocab_size, max_len=128).to(device)
ids = torch.randint(0, vocab_size, (2, 16), device=device)
logits, _ = m(ids)
logits.shape

## 3. KV-cache generation skeleton

In training, you feed full sequences and do not use KV-cache.
In generation, you typically:
- run full prompt once to initialize cache
- then feed one token at a time, using cache for speed

In [ ]:

@torch.inference_mode()
def gpt_generate(model, prompt_ids, max_new=50, temperature=1.0, top_k=50):
    model.eval()
    x = prompt_ids.unsqueeze(0)  # [1,T]
    # initialize cache with full prompt
    logits, cache = model(x, kv_cache=[(None,None)]*len(model.blocks))  # placeholder, will be handled below

The fully correct KV-cache interface is model-specific and requires careful handling of the initial cache.
Use this notebook as a blueprint for the architecture pieces (RMSNorm, RoPE, SwiGLU, causal attention).
For practical chatbot building, it is standard to rely on Hugging Face generation which already includes KV-cache support.